# A custom resource releases its handle, not its data

The external storage is represented by a list supplied by the caller. The resource controls access to it for one call.

Use the repository Python environment and run the cells in order. This notebook has its own setup; no previous notebook is required.

## 1. Import the implementation

The lifecycle code comes from AOA itself. The example supplies only the resource or external connection used to observe it.

In [ ]:
!pip install -q aoa-action-machine

In [2]:
import asyncio

from aoa.action_machine.resources import BaseStorage
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.entities import entities
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.resources.base_resource import BaseResource
from aoa.action_machine.resources.connection_scope import connection_scope
from aoa.action_machine.resources.per_call_connection import PerCallConnection

## 2. Define the objects used in this case

These definitions give the case an observable result without introducing unrelated business behavior.

In [3]:
class NotesDomain(BaseDomain):
    """Group the custom resource example."""
    name = "notes"
    description = "Notes transport"

@meta(description="Append notes to a supplied store")
@entities()
class NotesStorage(BaseStorage):
    """Expose storage access with an explicit usable interval."""
    def __init__(self, storage, events):
        """Remember the external storage and lifecycle log."""
        self.storage = storage
        self.events = events
        self.opened = False
    @classmethod
    def get_wrapper_class(cls):
        """Allow direct use of this non-transactional teaching store."""
        return None
    async def open(self):
        """Make the handle usable."""
        self.opened = True
        self.events.append("open")
    async def append(self, note):
        """Write through the open handle."""
        if not self.opened:
            raise RuntimeError("Resource is not open")
        self.storage.append(note)
        return len(self.storage)
    async def close(self):
        """Release this handle without deleting stored data."""
        self.opened = False
        self.events.append("close")

## 3. Prepare this call

Create the state and declarations used below. Merely declaring a per-call factory does not run it.

In [4]:
storage = []
events = []

## 4. Run and check the observation

The assertions test this case’s stated behavior. Printed output is recorded below; expected failures are caught explicitly.

In [5]:
async with connection_scope({"notes": PerCallConnection(
    lambda: NotesStorage(storage, events), lambda resource: resource.close(),
)}) as connections:
    count = await connections["notes"].append("hello")
    print("written:", count)
assert storage == ["hello"]
assert events == ["open", "close"]
print("stored after close:", storage)
print(events)

written: 1
stored after close: ['hello']
['open', 'close']


## 5. Explain the result

The resource is closed but the data remains in the supplied store. In a real implementation, close releases a connection or file handle; it does not erase the external data. A transactional resource should return a wrapper that also prevents child actions from closing its owner.